In [1]:
from db import get_con
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scipy.optimize import curve_fit


con = get_con()

Setting up database tables...
Done. All tables are ready.


# Описание задачи
<br>
Мы закупили установки мобильного приложения.
<br><br>
Есть 3 таблицы:

- $installs$ (файл task_2_installs.csv) - установки приложения
    - $user\_id$ - id юзера,
    - $install\_ts$ - время установки,
    - $campaign\_id$ - id рекламной кампании, приведшей юзера.
<br><br>
* $costs$ (файл task_2_costs.csv) - траты на рекламные кампании
    - $campaign\_id$ - id кампании,
    - $pdate$ - дата трат,
    - $cost$ - сумма трат в USD.
<br><br>
* $purchases$ (файл task_2_purchases.csv) - покупки юзеров в мобильном приложении
    - $user\_id$ - id юзера,
    - $purchase\_ts$ - время совершения покупки,
    - $revenue$ - сумма покупки в USD.
<br><br>

Вопросы:
1. Посчитайте общий накопленный ROAS (отношение дохода к тратам) в динамике количества дней с момента установки приложения.
2. Постройте простой прогноз ROAS на 365 дней жизни в приложении.

# Краткое описание решения и заметки

Глобально, мы считаем метрику следующим образом:
$$
ROAS_N =
\frac{\text{накопленная выручка до дня }N}
{\text{общие рекламные затраты}}
$$
Фактически, ROAS показывает, какая доля рекламных затрат окупилась накопленной выручкой к соответствующему дню жизни пользователя.

Я хочу, чтобы итоговая таблица для нашего первого вопроса выглядела +- так.
| День жизни | Накопленная выручка, USD | Затраты, USD | Накопленный ROAS |
|---:|---:|---:|---:|
| D0  | 458.67  | 4,558.74 | 10.06% |
| D7  | 855.51  | 4,558.74 | 18.77% |
| D14 | 1,106.27 | 4,558.74 | 24.27% |
| D29 | 1,285.23 | 4,558.74 | 28.19% |

Дальше вопрос, как мы посчитаем ее детально.
1. Базовый сценарий: считаем тотал пользователей, их выручку, общие траты на рекламу.
2. Берем усредненное значение, например: недельная когорта | campaign. Затем считаем тотал и усредняем.

Базово, мне бы хотелось идти через детализированный сценарий, но учитывая, что у нас мало данных + у нас большой хвост мелких кампаний — я бы тут не стала идти через усреднение.

Также важно по тому, как мы считаем N-ый день. У нас есть вариант считать календарный день, а есть вариант — считать прохождение 24 часов с момента установки.

Учитывая, что в задании есть про момент установки — возьмем 24 часа.

# Изучение данных

In [2]:
con.sql(
    """
    select campaign_id,
           min(date(install_ts)),
           max(date(install_ts))
    from task_2_installs
    group by campaign_id
    """).df()

,campaign_id,min(CAST(install_ts AS DATE)),max(CAST(install_ts AS DATE))
0,18529088791,2022-11-03,2022-11-30
1,18340470460,2022-11-04,2022-11-30
2,15010179776,2022-11-03,2022-11-18
3,18796908459,2022-11-02,2022-11-30
4,18725479090,2022-11-03,2022-11-30
5,15692110275,2022-11-03,2022-11-18
6,18869995641,2022-11-11,2022-11-24
7,15692110281,2022-11-05,2022-11-19
8,17497274587,2022-11-15,2022-11-21
9,18870069966,2022-11-10,2022-11-10


Изучим доступные нам данные. Начнем с данных в task_2_install.
У нас здесь связки user_id — campaign_id.

1. Есть кейс user_id = '0a99d7713d211df420d047782a33bf1e'. На одного и того же пользователя 2 даты регистрации. При этом campaign_id один и тот же, но одинаковый ID создает дубликат. Я возьму first_value по install_ts
2. Данные содержат информацию о 14ти компаниях.
3. Данные с 2022-11-02 до 2022-11-30
4. При этом некоторые компании привлекли установки только 1 день (пример: 18870069966), по некоторым кампаниям — установки есть в течение всего периода.

In [3]:
con.sql(
    """
    select count(*),
           count(distinct concat(cast(campaign_id as string),
                                 ' | ',
                                 cast(pdate as string))),
           count(concat(cast(campaign_id as string),
                                 ' | ',
                        cast(pdate as string)))

    from task_2_costs
    """).df()

,count_star(),"count(DISTINCT concat(CAST(campaign_id AS VARCHAR), ' | ', CAST(pdate AS VARCHAR)))","count(concat(CAST(campaign_id AS VARCHAR), ' | ', CAST(pdate AS VARCHAR)))"
0,192,192,192


Эта табличка содержит агрегированные косты на каждый день кампании. Здесь нет дублей, строго 1 строчка = 1 день = 1 кампания.

In [4]:
con.sql(
    """
    with cte_costs_aggrs as
        (select campaign_id,
                pdate,
                sum(cost) as cost
         from task_2_costs
         group by 1,2)

        , cte_installs_aggrs as
        (select campaign_id,
                date(install_ts) as pdate,
                count(distinct user_id) as cnt_users
         from task_2_installs
         group by 1,2)

    select *
    from cte_installs_aggrs inst
    left join cte_costs_aggrs csts
           on inst.campaign_id = csts.campaign_id
              and date(inst.pdate) = date(csts.pdate)
    """).df()

,campaign_id,pdate,cnt_users,campaign_id_1,pdate_1,cost
0,18796908459,2022-11-11,18,18796908459,2022-11-11,7.121223
1,18340470460,2022-11-30,8,18340470460,2022-11-30,4.897180
2,18725479090,2022-11-18,57,18725479090,2022-11-18,50.297472
3,18725479090,2022-11-25,39,18725479090,2022-11-25,48.458683
4,18796908459,2022-11-16,19,18796908459,2022-11-16,14.848264
...,...,...,...,...,...,...
187,18340470460,2022-11-11,5,18340470460,2022-11-11,0.560751
188,14883589082,2022-11-11,1,14883589082,2022-11-11,35.320000
189,18796908459,2022-11-04,21,18796908459,2022-11-04,14.922096
190,15692110281,2022-11-19,1,15692110281,2022-11-19,8.430000


Для каждой кампании из costs — есть установки.
Для каждой кампании, которая привлекла установки, есть косты.

При этом это соотновится, даже если добавить к join именно даты.

In [5]:
con.sql(
    """
    select *
    from task_2_purchases
    """).df()

,user_id,purchase_ts,revenue
0,ca98e36115b62b26813347438dd6734a,2022-11-15 07:12:25.819192,2.06
1,55b2f341992612aa3bf3b5757065bd85,2022-11-23 23:54:37.095631,6.82
2,4dc2bb839441a865d8f03a0e0fb47758,2022-11-21 14:40:42.565536,0.00
3,cd10f5b6e13cb51b159b8e6bc37d3362,2022-11-29 12:09:10.853598,0.00
4,cd10f5b6e13cb51b159b8e6bc37d3362,2022-11-29 02:51:10.317947,3.70
...,...,...,...
287,841499ff70f20787886cccecb5a77953,2022-12-02 15:26:59.627381,0.00
288,be62570f7330c20a61e57098d36bef6f,2022-11-26 04:08:24.946969,0.36
289,ede8029aefac4cca04095f0eec394151,2022-11-07 08:47:46.047042,4.48
290,56c88b6f7b494456a8e3729207436c93,2022-11-17 13:30:47.743987,5.66


В данных о покупках все в целом тоже предельно просто. user_id — дата покупки — сумма покупки. На одного пользователя приходится несколько покупок. Покупки покрывают более широкий период — до 2022-12-29.

При этом есть строчки, где revenue = 0.00

И опять же, записей с несколькими 0.00 revenue — может быть несколько. Это странно.

In [6]:
con.sql(
    """
    with cte_purchases_aggrs as
        (select user_id,
                min(purchase_ts) as first_pusrchase_date,
                count(*) as cnt_purchases,
                sum(revenue) as revenue
         from task_2_purchases
         group by user_id)

        , cte_installs_aggrs as
        (select user_id,
                min(install_ts) as first_install_date
         from task_2_installs
         group by user_id)

        select *
        from cte_installs_aggrs inst
        left join cte_purchases_aggrs prch
                on inst.user_id = prch.user_id
        where date(first_pusrchase_date) = date(first_install_date)
    """).df()

,user_id,first_install_date,user_id_1,first_pusrchase_date,cnt_purchases,revenue
0,5f3979b28ccaaba4163431f8a9819f2e,2022-11-23 08:22:57.617,5f3979b28ccaaba4163431f8a9819f2e,2022-11-23 08:27:13.482458,1,1.55
1,9a51e649f53c5ab5befe68fee1e79ae1,2022-11-09 04:53:19.455,9a51e649f53c5ab5befe68fee1e79ae1,2022-11-09 05:09:17.169644,1,10.45
2,79c5dd33af227b55eacffe01e18186c6,2022-11-16 10:29:42.212,79c5dd33af227b55eacffe01e18186c6,2022-11-16 11:01:28.324681,2,10.32
3,ea0712aec004607b9119c96e008a4a97,2022-11-27 06:35:36.831,ea0712aec004607b9119c96e008a4a97,2022-11-27 06:45:09.473985,4,0.00
4,2c77c2a71338af48a12c5862e346207a,2022-11-26 05:35:43.815,2c77c2a71338af48a12c5862e346207a,2022-11-26 05:38:24.204307,1,1.82
5,23f19069737d0bfc6ef8000979f133f2,2022-11-26 12:33:26.026,23f19069737d0bfc6ef8000979f133f2,2022-11-26 12:40:35.753484,6,4.83
6,c613dd649cab391ac6f4bf7cd78bfcfe,2022-11-11 14:51:17.975,c613dd649cab391ac6f4bf7cd78bfcfe,2022-11-11 15:59:09.889579,1,2.11
7,239f394f56bb771d599fa64f94ae8b91,2022-11-21 10:58:31.518,239f394f56bb771d599fa64f94ae8b91,2022-11-21 13:13:41.665320,2,10.04
8,48cc938ccf2fc5f93312b221c64aacdf,2022-11-20 17:49:04.043,48cc938ccf2fc5f93312b221c64aacdf,2022-11-20 17:51:29.439057,1,2.27
9,62d081ef0d04c2a24932f5b714557798,2022-11-15 16:16:34.613,62d081ef0d04c2a24932f5b714557798,2022-11-15 18:57:32.824158,1,0.39


Установки идут по 2022-11-30, а покупки — по 2022-12-29. Прежде чем считать кривую, нужно понять, чем обрезаны данные о покупках: календарной датой или возрастом юзера. От этого зависит, есть ли правый цензор (у ноябрьских когорт разный запас календарного времени) и до какого дня кривой можно доверять. Проверим максимальный день жизни в разрезе даты установки.

In [7]:
con.sql(
    """
    with installs as (
        select user_id,
               min(install_ts) as install_ts
        from task_2_installs
        group by user_id
    )

    select date(i.install_ts) as install_date,
           count(distinct p.user_id) as cnt_users,
           max(cast(floor(date_diff('second', i.install_ts, p.purchase_ts) / 86400.0) as int)) as max_lifetime_day,
           max(date(p.purchase_ts)) as max_purchase_date
    from installs i
    join task_2_purchases p
      on p.user_id = i.user_id
    group by 1
    order by 1
    """).df()

,install_date,cnt_users,max_lifetime_day,max_purchase_date
0,2022-11-03,1,0,2022-11-03
1,2022-11-04,4,27,2022-12-02
2,2022-11-05,7,25,2022-12-01
3,2022-11-06,6,25,2022-12-01
4,2022-11-07,1,8,2022-11-16
5,2022-11-08,7,15,2022-11-24
6,2022-11-09,6,13,2022-11-23
7,2022-11-10,4,15,2022-11-26
8,2022-11-11,8,11,2022-11-23
9,2022-11-12,9,20,2022-12-02


У каждой когорты установки максимальный день жизни не превышает 29, а у поздних когорт (24, 26, 28, 30 ноября) он равен ровно 29. По 28 когортам это не может быть совпадением: данные о покупках обрезаны **по возрасту юзера — ровно 30 дней жизни**, а не по календарной дате.

Это важно, потому что это определенным образом дает возможность нам строить запрос.

1. Во-первых, лишний раз подтверждаем, что считаем НЕ календарные дни, а время с момента установки.
2. Во-вторых, нам не нужно самим обрезать данные. Они явно обрезаны и когорты уравнены.

# Основной код

## Вопрос 1 — считаем фактический ROAS

### Запрос для подсчета Total ROAS, без разбивки по дате установки и campaign_id

In [8]:
df_roas_total = con.sql(
    """
    with installs as (
        select
            user_id,
            install_ts,
            cast(install_ts as date) as install_date,
            campaign_id
        from task_2_installs
        qualify row_number() over (
            partition by user_id
            order by install_ts
        ) = 1 -- В данном случае, обрабатываем конкретный кейс с задублированный user_id
    ),

    revenue as (
        select
            cast(
                floor(
                    date_diff('second', i.install_ts, p.purchase_ts)
                    / 86400.0
                ) as integer
            ) as lifetime_day,
            p.revenue
        from installs i
        join task_2_purchases p
          on p.user_id = i.user_id
        where p.purchase_ts >= i.install_ts -- Больше формальная приписка. Мы изучали данные и там нет таких примеров
    ),

    daily_revenue as (
        select
            lifetime_day,
            sum(revenue) as revenue
        from revenue
        where lifetime_day between 0 and 29 -- Опять же, более формальная приписка.
        -- У нас нет в данных примеров за пределами 29 дней
        group by lifetime_day
    ),

    days as (
        select unnest(generate_series(0, 29)) as lifetime_day
    ),

    curve as (
        select
            d.lifetime_day,
            coalesce(r.revenue, 0) as daily_revenue
        from days d
        left join daily_revenue r using (lifetime_day)
    ),

    cumulative as (
        select
            lifetime_day,
            sum(daily_revenue) over (
                order by lifetime_day
                rows between unbounded preceding and current row
            ) as cum_revenue
        from curve
    ),

    total_cost as (
        select sum(cost) as cost
        from task_2_costs
    )

    select
        c.lifetime_day as days_since_install,
        round(c.cum_revenue, 2) as cum_revenue,
        round(tc.cost, 2) as total_cost,
        round(100 * c.cum_revenue / nullif(tc.cost, 0), 2) as cum_roas_pct
    from cumulative c
    cross join total_cost tc
    order by days_since_install
    """
).df()

In [9]:
df_roas_total

,days_since_install,cum_revenue,total_cost,cum_roas_pct
0,0,458.67,4558.74,10.06
1,1,556.26,4558.74,12.20
2,2,663.26,4558.74,14.55
3,3,682.31,4558.74,14.97
4,4,720.58,4558.74,15.81
5,5,752.81,4558.74,16.51
6,6,813.19,4558.74,17.84
7,7,855.51,4558.74,18.77
8,8,893.92,4558.74,19.61
9,9,893.92,4558.74,19.61


In [10]:
fig = px.line(
    df_roas_total,
    x="days_since_install",
    y="cum_roas_pct",
    title="ROAS",
    labels={"days_since_install": "День с момента установки",
            "cum_roas_pct": "ROAS (%)"}
)
fig.show()

### Запрос для подсчета ROAS, уже добавляя среднее по campaign_id и дате установки

Добавим одновременно:
1. Среднюю накопленную выручку на установку;
2. Обычный средний ROAS когорты;
3. Медианный ROAS когорты;
4. Взвешенный средний ROAS — основной бизнес-показатель.

In [11]:
roas_by_lifetime_day = con.sql(
    """
    with installs as (
        -- Первая установка каждого пользователя
        select
            user_id,
            install_ts,
            cast(install_ts as date) as install_date,
            campaign_id
        from task_2_installs
        qualify row_number() over (
            partition by user_id
            order by install_ts
        ) = 1
    ),

    install_cohorts as (
        -- Размер каждой когорты: кампания × дата установки
        select
            campaign_id,
            install_date,
            count(*) as installs
        from installs
        group by 1, 2
    ),

    cohort_costs as (
        -- Затраты на каждую когорту
        select
            campaign_id,
            cast(pdate as date) as install_date,
            sum(cost) as cost
        from task_2_costs
        group by 1, 2
    ),

    purchases as (
        -- Покупки с кампанией, когортой и днём жизни
        select
            i.campaign_id,
            i.install_date,
            cast(
                floor(
                    date_diff(
                        'second',
                        i.install_ts,
                        p.purchase_ts
                    ) / 86400.0
                ) as integer
            ) as lifetime_day,
            p.revenue
        from installs i
        join task_2_purchases p
          on p.user_id = i.user_id
        where p.purchase_ts >= i.install_ts
    ),

    daily_revenue as (
        -- Выручка каждой когорты в отдельный день жизни
        select
            campaign_id,
            install_date,
            lifetime_day,
            sum(revenue) as daily_revenue
        from purchases
        where lifetime_day between 0 and 29
        group by 1, 2, 3
    ),

    days as (
        select unnest(generate_series(0, 29)) as lifetime_day
    ),

    cohort_grid as (
        -- Каждая когорта получает полную сетку D0–D29
        select
            i.campaign_id,
            i.install_date,
            i.installs,
            c.cost,
            d.lifetime_day
        from install_cohorts i
        join cohort_costs c
          on c.campaign_id = i.campaign_id
         and c.install_date = i.install_date
        cross join days d
    ),

    cohort_curve as (
        -- Накопленная выручка внутри каждой когорты
        select
            g.campaign_id,
            g.install_date,
            g.installs,
            g.cost,
            g.lifetime_day,

            sum(coalesce(r.daily_revenue, 0)) over (
                partition by g.campaign_id, g.install_date
                order by g.lifetime_day
                rows between unbounded preceding and current row
            ) as cum_revenue

        from cohort_grid g
        left join daily_revenue r
          on r.campaign_id = g.campaign_id
         and r.install_date = g.install_date
         and r.lifetime_day = g.lifetime_day
    ),

    cohort_metrics as (
        select
            campaign_id,
            install_date,
            lifetime_day,
            installs,
            cost,
            cum_revenue,

            -- LTV/ARPU: накопленная выручка на одну установку
            cum_revenue / nullif(installs, 0)
                as cum_revenue_per_install,

            -- ROAS конкретной когорты
            100 * cum_revenue / nullif(cost, 0)
                as cohort_roas_pct

        from cohort_curve
    )

    select
        lifetime_day as days_since_install,

        -- Среднее LTV, если каждой когорте дать одинаковый вес
        round(
            avg(cum_revenue_per_install),
            4
        ) as avg_cohort_revenue_per_install,

        -- Среднее LTV по всем пользователям
        round(
            sum(cum_revenue) / nullif(sum(installs), 0),
            4
        ) as weighted_avg_revenue_per_install,

        -- Обычное среднее ROAS когорт
        round(
            avg(cohort_roas_pct),
            2
        ) as avg_cohort_roas_pct,

        -- ROAS типичной когорты, устойчивее к выбросам
        round(
            median(cohort_roas_pct),
            2
        ) as median_cohort_roas_pct,

        -- Основной показатель:
        -- сумма выручки / сумма затрат
        round(
            100 * sum(cum_revenue) / nullif(sum(cost), 0),
            2
        ) as weighted_avg_roas_pct,

        count(*) as cohorts_count

    from cohort_metrics
    group by lifetime_day
    order by lifetime_day
    """
).df()

In [12]:
roas_by_lifetime_day

,days_since_install,avg_cohort_revenue_per_install,weighted_avg_revenue_per_install,avg_cohort_roas_pct,median_cohort_roas_pct,weighted_avg_roas_pct,cohorts_count
0,0,0.0730,0.0855,13.04,0.0,10.06,192
1,1,0.0860,0.1037,15.41,0.0,12.20,192
2,2,0.0999,0.1237,17.24,0.0,14.55,192
3,3,0.1033,0.1272,20.05,0.0,14.97,192
4,4,0.1069,0.1343,20.54,0.0,15.81,192
5,5,0.1101,0.1403,21.23,0.0,16.51,192
6,6,0.1196,0.1516,22.57,0.0,17.84,192
7,7,0.1292,0.1595,27.50,0.0,18.77,192
8,8,0.1354,0.1667,28.91,0.0,19.61,192
9,9,0.1354,0.1667,28.91,0.0,19.61,192


In [13]:
fig = px.line(
    roas_by_lifetime_day,
    x="days_since_install",
    y=["avg_cohort_roas_pct",
       "median_cohort_roas_pct",
       "weighted_avg_roas_pct"],
    title="ROAS"
)
fig.show()

График получился очень показательный — и он демонстрирует, почему обычный avg здесь вводит в заблуждение.

1. График среднего достаточно сильно завышает общие показатели. Каждая когорта campaign × day имеет одинаковый вес, даже если в одной потратили 2, а в другой 500. Несколько маленьких успешных когорт сильно завышают среднее.
2. Медиана равно 0, в целом, это логично, потому что большое количество дневных коготр не получила выручки.
3. Взвешенный график — аналог того, что мы считали ДО. По факту, это сумма выручки на сумму костов. И она дает достаточно неплохой результат.

В целом, можно поупарываться дальше. Взять, например, не дневные, а недельные когорты. Но скорее всего, для этой задачи это оверкил.

## Вопрос 2 — Построение прогноза

Для прогноза накопленного ROAS используются три простые модели. Они отражают разные предположения о том, как будет формироваться выручка после доступного периода D0–D29.

##### Экспоненциальное насыщение

$$
ROAS(d) = L - A e^{-kd}
$$

где:

- $L$ — предельный уровень ROAS;
- $A$ — расстояние от начального значения до предельного уровня;
- $k$ — скорость выхода на плато;
- $d$ — день жизни пользователя.

Модель предполагает, что основная часть выручки появляется в начале жизни пользователя, после чего ежедневный прирост быстро уменьшается. Со временем накопленный ROAS выходит на конечное плато $L$.

Эта модель соответствует консервативному сценарию: после первого месяца пользователи приносят мало дополнительной выручки.

##### Логарифмическая модель

$$
ROAS(d) = a + b\ln(1+d)
$$

где:

- $a$ — базовый уровень ROAS;
- $b$ — интенсивность последующего роста;
- $d$ — день жизни пользователя.

Модель также предполагает постепенное замедление роста, но не фиксирует конечное плато. Пользователи продолжают приносить дополнительную выручку, однако её прирост с каждым днём становится меньше.

Этот вариант можно интерпретировать как умеренный сценарий с долгосрочным хвостом повторных покупок.

##### Степенная модель

$$
ROAS(d) = a + b(d+1)^p, \qquad 0 < p < 1
$$

где:

- $a$ — базовый уровень;
- $b$ — масштаб роста;
- $p$ — скорость затухания;
- $d$ — день жизни пользователя.

При $0<p<1$ накопленный ROAS продолжает расти, но скорость роста постепенно снижается. По сравнению с другими моделями степенная функция предполагает наиболее длинный хвост будущей выручки.

Это оптимистичный сценарий, при котором пользователи могут возвращаться в приложение и совершать покупки спустя несколько месяцев.

##### Почему выбраны эти модели

Рассматриваемые функции подходят для накопленного ROAS, поскольку:

- накопленный ROAS должен быть неубывающим;
- основная выручка обычно формируется в начале жизни пользователя;
- последующий прирост постепенно замедляется;
- модели содержат мало параметров и могут быть обучены на небольшом количестве наблюдений;
- три модели позволяют проверить несколько вариантов поведения долгосрочного хвоста.

Линейная модель не используется, поскольку она предполагает одинаковый прирост ROAS каждый день и может дать нереалистично высокий прогноз на D365.

Исходя из этого, получаем следующую логику результата:
1. Обучаем все три модели на всех фактических данных с 0 до 29 дня и прогнозируем до 365 дня
2. Отдельно проверяем прогнозы, берем до 21 дня, а строим прогноз до 29
3. Выводим результат

### Подготавливаем данные

In [14]:
actual = (
    roas_by_lifetime_day[
        ["days_since_install", "weighted_avg_roas_pct"]
    ]
    .dropna()
    .drop_duplicates("days_since_install")
    .sort_values("days_since_install")
    .rename(columns={"weighted_avg_roas_pct": "actual_roas_pct"})
    .reset_index(drop=True)
)

x_actual = actual["days_since_install"].to_numpy(dtype=float)
y_actual = actual["actual_roas_pct"].to_numpy(dtype=float)

LAST_ACTUAL_DAY = int(x_actual.max())

# D0–D20 — это первые 21 дня жизни.
# Прогноз в бэктесте строим для D21–D29.
BACKTEST_END_DAY = 20
FORECAST_END_DAY = 365

### Определение моделей

In [15]:
def exponential_curve(day, ceiling, amplitude, decay_rate):
    """
    Накопленный ROAS постепенно выходит на плато ceiling.
    """
    return ceiling - amplitude * np.exp(-decay_rate * day)

In [16]:
def power_curve(day, intercept, scale, power):
    """
    Степенная модель: рост замедляется, но не имеет фиксированного плато.
    """
    return intercept + scale * (day + 1) ** power

In [17]:
def fit_models(x, y):
    """
    Обучает три модели:
    1. Exponential saturation
    2. Logarithmic
    3. Power law
    """

    # Экспоненциальное насыщение
    initial_ceiling = y.max() * 1.2
    initial_amplitude = initial_ceiling - y[0]

    exp_params, _ = curve_fit(
        exponential_curve,
        x,
        y,
        p0=[initial_ceiling, initial_amplitude, 0.08],
        bounds=(
            [y.max(), 0, 0.000001],
            [200, 200, 2]
        ),
        maxfev=100_000
    )

    # Логарифмическая модель:
    # y = intercept + slope × log(1 + day)
    log_slope, log_intercept = np.polyfit(
        np.log1p(x),
        y,
        deg=1
    )

    # Степенная модель
    power_params, _ = curve_fit(
        power_curve,
        x,
        y,
        p0=[0, 10, 0.3],
        bounds=(
            [-100, 0, 0.01],
            [100, 200, 1]
        ),
        maxfev=100_000
    )

    return {
        "exponential": {
            "label": "Экспоненциальное насыщение",
            "params": exp_params
        },
        "logarithmic": {
            "label": "Логарифмическая модель",
            "params": np.array([log_intercept, log_slope])
        },
        "power": {
            "label": "Степенная модель",
            "params": power_params
        }
    }

In [18]:
def predict_model(model_name, params, days):
    days = np.asarray(days, dtype=float)

    if model_name == "exponential":
        return exponential_curve(days, *params)

    if model_name == "logarithmic":
        intercept, slope = params
        return intercept + slope * np.log1p(days)

    if model_name == "power":
        return power_curve(days, *params)

    raise ValueError(f"Неизвестная модель: {model_name}")

### Прогноз D365: обучение на D0–D29

In [19]:
full_models = fit_models(x_actual, y_actual)
all_days = np.arange(0, FORECAST_END_DAY + 1)

full_forecast = pd.DataFrame({
    "days_since_install": all_days
})

for model_name, model in full_models.items():
    full_forecast[f"{model_name}_roas_pct"] = predict_model(
        model_name,
        model["params"],
        all_days
    )

# Добавляем фактические значения
full_forecast = full_forecast.merge(
    actual,
    on="days_since_install",
    how="left"
)

forecast_d365 = pd.DataFrame([
    {
        "model": model["label"],
        "roas_d365_pct": predict_model(
            model_name,
            model["params"],
            [FORECAST_END_DAY]
        )[0]
    }
    for model_name, model in full_models.items()
]).sort_values("roas_d365_pct")

In [20]:
forecast_d365

,model,roas_d365_pct
0,Экспоненциальное насыщение,30.267625
1,Логарифмическая модель,42.597338
2,Степенная модель,61.608066


### Бэктест

In [21]:
train = actual[
    actual["days_since_install"] <= BACKTEST_END_DAY
].copy()

test = actual[
    actual["days_since_install"] > BACKTEST_END_DAY
].copy()

x_train = train["days_since_install"].to_numpy(dtype=float)
y_train = train["actual_roas_pct"].to_numpy(dtype=float)

x_test = test["days_since_install"].to_numpy(dtype=float)
y_test = test["actual_roas_pct"].to_numpy(dtype=float)

backtest_models = fit_models(x_train, y_train)

backtest_forecast = actual.copy()
backtest_metrics = []

for model_name, model in backtest_models.items():
    column_name = f"{model_name}_backtest_roas_pct"

    backtest_forecast[column_name] = predict_model(
        model_name,
        model["params"],
        backtest_forecast["days_since_install"]
    )

    y_pred = predict_model(
        model_name,
        model["params"],
        x_test
    )

    errors = y_pred - y_test

    backtest_metrics.append({
        "model": model["label"],
        "mae_pct_points": np.mean(np.abs(errors)),
        "rmse_pct_points": np.sqrt(np.mean(errors ** 2)),
        "bias_pct_points": np.mean(errors),
        "predicted_roas_d29_pct": y_pred[-1],
        "actual_roas_d29_pct": y_test[-1]
    })

backtest_metrics = (
    pd.DataFrame(backtest_metrics)
    .sort_values("rmse_pct_points")
    .reset_index(drop=True)
)

backtest_metrics.round(2)

,model,mae_pct_points,rmse_pct_points,bias_pct_points,predicted_roas_d29_pct,actual_roas_d29_pct
0,Экспоненциальное насыщение,0.38,0.48,-0.35,27.66,28.19
1,Степенная модель,0.86,0.92,0.86,29.63,28.19
2,Логарифмическая модель,1.07,1.11,-1.07,27.04,28.19


### Визуализация результатов

In [22]:
colors = {
    "exponential": "#636EFA",
    "logarithmic": "#AB63FA",
    "power": "#FFA15A"
}

fig = make_subplots(
    rows=2,
    cols=1,
    vertical_spacing=0.13,
    subplot_titles=(
        "Прогноз ROAS D365 на данных D0–D29",
        "Бэктест: обучение D0–D20, прогноз D21–D29"
    )
)


# Верхний график: полный прогноз D365
fig.add_trace(
    go.Scatter(
        x=actual["days_since_install"],
        y=actual["actual_roas_pct"],
        mode="lines+markers",
        name="Фактический ROAS",
        line=dict(color="#00CC96", width=4),
        marker=dict(size=5)
    ),
    row=1,
    col=1
)

future_mask = (
    full_forecast["days_since_install"] >= LAST_ACTUAL_DAY
)

for model_name, model in full_models.items():
    fig.add_trace(
        go.Scatter(
            x=full_forecast.loc[
                future_mask,
                "days_since_install"
            ],
            y=full_forecast.loc[
                future_mask,
                f"{model_name}_roas_pct"
            ],
            mode="lines",
            name=model["label"],
            line=dict(
                color=colors[model_name],
                width=3,
                dash="dash"
            )
        ),
        row=1,
        col=1
    )


# Нижний график: обучение и проверочная выборка
fig.add_trace(
    go.Scatter(
        x=train["days_since_install"],
        y=train["actual_roas_pct"],
        mode="lines+markers",
        name="Обучающий факт D0–D20",
        line=dict(color="#00CC96", width=4),
        marker=dict(size=6),
        showlegend=False
    ),
    row=2,
    col=1
)

# Добавляем D20, чтобы линия факта была непрерывной
test_for_plot = actual[
    actual["days_since_install"] >= BACKTEST_END_DAY
]

fig.add_trace(
    go.Scatter(
        x=test_for_plot["days_since_install"],
        y=test_for_plot["actual_roas_pct"],
        mode="lines+markers",
        name="Отложенный факт D21–D29",
        line=dict(color="#FFFFFF", width=4),
        marker=dict(size=7),
        showlegend=False
    ),
    row=2,
    col=1
)

backtest_plot_mask = (
    backtest_forecast["days_since_install"]
    >= BACKTEST_END_DAY
)

for model_name, model in backtest_models.items():
    fig.add_trace(
        go.Scatter(
            x=backtest_forecast.loc[
                backtest_plot_mask,
                "days_since_install"
            ],
            y=backtest_forecast.loc[
                backtest_plot_mask,
                f"{model_name}_backtest_roas_pct"
            ],
            mode="lines",
            name=f"Бэктест: {model['label']}",
            line=dict(
                color=colors[model_name],
                width=3,
                dash="dash"
            ),
            showlegend=False
        ),
        row=2,
        col=1
    )


fig.add_hline(
    y=100,
    line_dash="dot",
    line_color="#EF553B",
    annotation_text="Окупаемость 100%",
    row=1,
    col=1
)

fig.add_vline(
    x=LAST_ACTUAL_DAY,
    line_dash="dot",
    line_color="#AAAAAA",
    annotation_text="Начало прогноза",
    row=1,
    col=1
)

fig.add_vline(
    x=BACKTEST_END_DAY,
    line_dash="dot",
    line_color="#AAAAAA",
    annotation_text="Граница обучения",
    row=2,
    col=1
)

fig.update_xaxes(
    title_text="Дней с момента установки",
    row=1,
    col=1
)

fig.update_xaxes(
    title_text="Дней с момента установки",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Накопленный ROAS, %",
    row=1,
    col=1
)

fig.update_yaxes(
    title_text="Накопленный ROAS, %",
    row=2,
    col=1
)

fig.update_layout(
    title="Прогноз накопленного ROAS и историческая проверка моделей",
    template="plotly_dark",
    height=950,
    hovermode="x unified",
    legend_title_text="Модель"
)

fig.show()